# 🚀 Unity Catalog

---

## 🔷 1. What is Unity Catalog?

Unity Catalog is a **centralized governance layer in Databricks** used to manage:

- 🔐 Data Security (who can access what)
- 🧠 Metadata (tables, schemas, catalogs)
- 🔍 Data Lineage (data flow tracking)
- 📜 Auditing (who did what)

👉 Simple Definition:
> Unity Catalog = **Single place to control and secure all your data across workspaces**

---

## 🔷 2. Why Do We Need It?

### ❌ Problems Before (Hive Metastore)
- Separate metadata per workspace
- Weak security (only table-level)
- No lineage tracking
- Hard to manage at scale

### ✅ With Unity Catalog
- Centralized governance
- Fine-grained access control
- Cross-workspace data sharing
- Built-in lineage + auditing

---

## 🔷 3. Core Hierarchy (Most Important 🔥)
Metastore
| ---
└── Catalog
| ---
└── Schema (Database)
| ---
└── Table / View


### 🧠 Real Example:

company_metastore

└── sales

└── analytics

└── customers

#### Fully Qualified Name: sales.analytics.customers


---

## 🔷 4. Key Concepts Explained Simply

### 🔹 Metastore
- Top-level container
- Stores metadata only (not actual data)

---

### 🔹 Catalog
- Represents a business domain

👉 Examples:
- sales
- finance
- hr

---

### 🔹 Schema (Database)
- Logical grouping inside catalog

👉 Examples:
- raw
- curated
- analytics

---

### 🔹 Tables

#### ✅ Managed Table
- Databricks manages storage
- Easy to use

#### ✅ External Table
- Data stored in S3 / ADLS / GCS
- More control

---

## 🔷 5. Creating Objects (SQL + PySpark)

---

### 🔹 Create Catalog
```sql
CREATE CATALOG sales;
```python
spark.sql("CREATE CATALOG sales")

### Create Schema
- CREATE SCHEMA sales.analytics;
    - spark.sql("CREATE SCHEMA sales.analytics")

🔹 Create Table

`CREATE TABLE sales.analytics.customers (
  id INT,
  name STRING,
  country STRING,
  revenue DOUBLE
);`

- `df.write.saveAsTable("sales.analytics.customers")`

🔹 Create External Table

`CREATE TABLE sales.analytics.customers_ext
USING DELTA
**LOCATION 's3://company-data/customers/';**`

`df.write \
  .format("delta") \
  .option("path","s3://company-data/customers/") \
  .saveAsTable("sales.analytics.customers_ext")`

🔷 6. Security Model (RBAC)

🔹 Identity Types
- 👤 Users
- 👥 Groups (Recommended ✅)
- 🤖 Service Principals

🔹 Permissions
- Permission	Meaning
- USE CATALOG	Enter catalog
- USE SCHEMA	Enter schema
- SELECT	Read data
- MODIFY	Write data
- ALL PRIVILEGES	Full access

🔷 7. 🔥 MOST IMPORTANT CONCEPT — Access Flow

👉 Access is TOP-DOWN

- USE CATALOG → USE SCHEMA → TABLE ACCESS

❌ Wrong Way (Very Common Mistake)

`GRANT SELECT ON TABLE sales.analytics.customers TO analysts;`

👉 This fails because higher-level access is missing.

`✅ Correct Way

GRANT USE CATALOG ON CATALOG sales TO analysts;

GRANT USE SCHEMA ON SCHEMA sales.analytics TO analysts;

GRANT SELECT ON TABLE sales.analytics.customers TO analysts;

✅ PySpark Equivalent

spark.sql("GRANT USE CATALOG ON CATALOG sales TO analysts")

spark.sql("GRANT USE SCHEMA ON SCHEMA sales.analytics TO analysts")

spark.sql("GRANT SELECT ON TABLE sales.analytics.customers TO analysts")`

### 8. Advanced Grants
🔹 Multiple Permissions

`GRANT SELECT, MODIFY ON TABLE sales.analytics.customers TO data_engineers;`

🔹 Full Access

`GRANT ALL PRIVILEGES ON TABLE sales.analytics.customers TO admin_group;`

🔹 Revoke Access

`REVOKE SELECT ON TABLE sales.analytics.customers FROM analysts;`

🔹 Check Permissions

`SHOW GRANTS ON TABLE sales.analytics.customers;`

`SHOW GRANTS TO analysts;`

🔷 9. Ownership (Critical Concept)

`ALTER TABLE sales.analytics.customers OWNER TO admin_group;`

👉 Owner = Full control over object

🔷 10. Fine-Grained Security (Advanced 🔥)

🔹 Column-Level Security

`GRANT SELECT (id, name) ON TABLE sales.analytics.customers TO analysts;`

👉 Users see only selected columns

🔹 Row-Level Security
`CREATE FUNCTION filter_india AS (country STRING) -> country = 'India';`

`ALTER TABLE sales.analytics.customers SET ROW FILTER filter_india ON (country);`

👉 Only rows with country = India are visible

🔹 Data Masking

`CREATE FUNCTION mask_revenue AS (rev DOUBLE) ->

  CASE

    WHEN current_user() = 'admin' THEN rev

    ELSE NULL

  END;
`

`ALTER TABLE sales.analytics.customers

ALTER COLUMN revenue SET MASK mask_revenue;`

## 🔷 10. Data Masking (Continuation)

👉 Sensitive data hidden from non-admin users

---

## 🔷 11. External Storage Integration

### 🔹 Storage Credential

```sql
CREATE STORAGE CREDENTIAL my_cred
WITH IAM ROLE 'arn:aws:iam::123456789:role/my-role';
🔹 External Location
CREATE EXTERNAL LOCATION sales_data
URL 's3://company-data/sales/'
WITH STORAGE CREDENTIAL my_cred;
🔹 Grant Access
GRANT READ FILES
ON EXTERNAL LOCATION sales_data
TO analysts;
🔷 12. How Access Works Internally
User Query
   ↓
Identity Check
   ↓
Catalog Permission ✔
   ↓
Schema Permission ✔
   ↓
Table Permission ✔
   ↓
Access Granted ✅ / Denied ❌
   ↓
Audit Logged
🔷 13. Real-World Scenario
👨‍💼 Requirement
Analysts → Read-only
Engineers → Full access
✅ Solution
-- Catalog Access
GRANT USE CATALOG ON CATALOG sales TO analysts;
GRANT USE CATALOG ON CATALOG sales TO engineers;

-- Schema Access
GRANT USE SCHEMA ON SCHEMA sales.analytics TO analysts;
GRANT USE SCHEMA ON SCHEMA sales.analytics TO engineers;

-- Table Access
GRANT SELECT ON TABLE sales.analytics.customers TO analysts;

GRANT SELECT, MODIFY
ON TABLE sales.analytics.customers
TO engineers;
🔷 14. Best Practices (Production 🚀)
✔ Use domain-based catalogs (sales, finance)
✔ Use layered schemas (bronze, silver, gold)
✔ Always use groups (NOT individual users)
✔ Restrict raw data access
✔ Follow least privilege principle
🔷 15. Common Mistakes 🚨
❌ Missing USE SCHEMA
❌ Giving only table access
❌ Using individual users instead of groups
❌ Overusing ALL PRIVILEGES
❌ Misconfigured external locations
🔷 16. Unity Catalog vs Hive
Feature        Hive            Unity Catalog
-------------------------------------------
Scope          Workspace      Cross-workspace
Security       Basic          Fine-grained
Lineage        ❌              ✅
Governance     Weak           Strong
🔷 17. Interview-Level Understanding 🎯
💡 Q1: How does Unity Catalog enforce security?
- Validates user identity
- Checks permissions at:
    → Catalog
    → Schema
    → Table
- Grants or denies access
- Logs everything
💡 Q2: Most Important Concept?
Top-down permission model

If user lacks access at ANY level → ❌ Access Denied
💡 Q3: Managed vs External Table?
Managed  → Databricks controls storage
External → User controls storage
💡 Q4: Why Unity Catalog?
Centralized + secure + scalable governance
🔷 18. Final Mental Model 🧠
Can user:
✔ Access Catalog?
✔ Access Schema?
✔ Access Table?

If ANY answer = ❌ → Access Denied
🔷 19. Final Summary 🚀
Unity Catalog = Governance Layer
Centralized across workspaces
Uses RBAC for security

Required Flow:
USE CATALOG → USE SCHEMA → TABLE ACCESS

Supports:
- Fine-grained access
- Row filtering
- Column masking
- External storage
- Auditing & lineage